# 04 Model Training
Train churn classification models and compare performance using accuracy, classification reports, confusion matrices, and ROC AUC.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")


In [ ]:
DATA_PATH = "../data/raw/Telco_customer_churn.xlsx"

df = pd.read_excel(DATA_PATH)
df.head()


## Preprocessing
The same feature set and preprocessing pipeline from the feature engineering notebook is recreated here so this notebook can run independently.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

feature_cols = [
    'Senior Citizen', 'Partner', 'Dependents', 'Internet Service',
    'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
    'Contract', 'Paperless Billing', 'Payment Method', 'Tenure Months',
    'Monthly Charges', 'CLTV', 'Total Charges'
]

X = df[feature_cols].copy()
y = df['Churn Value']

X['Total Charges'] = pd.to_numeric(X['Total Charges'], errors='coerce')

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore')),
        ('scaler', StandardScaler(with_mean=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, numerical_cols),
        ('cat_pipeline', cat_pipeline, categorical_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## Random Forest Baseline


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("
Classification Report:")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix')
plt.show()


## Compare Multiple Classification Models


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score

classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

try:
    import xgboost as xgb
    classifiers['XGBoost'] = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42
    )
except ImportError:
    print('xgboost is not installed, so XGBoost will be skipped.')

results = {}

for name, classifier in classifiers.items():
    print(f"===== Training {name} =====")

    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    y_pred_proba = classifier.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_pred_proba)

    results[name] = {
        'model': classifier,
        'accuracy': accuracy,
        'classification_report': report,
        'confusion_matrix': cm,
        'roc_auc_score': auc_score
    }

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {auc_score:.4f}")

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'{name} Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'{name} ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
comparison = pd.DataFrame({
    name: {
        'accuracy': metrics['accuracy'],
        'roc_auc_score': metrics['roc_auc_score'],
        'precision_churn': metrics['classification_report']['1']['precision'],
        'recall_churn': metrics['classification_report']['1']['recall'],
        'f1_churn': metrics['classification_report']['1']['f1-score']
    }
    for name, metrics in results.items()
}).T.sort_values('roc_auc_score', ascending=False)

comparison


In [ ]:
import joblib
from pathlib import Path

models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

best_model_name = comparison.index[0]
best_model = results[best_model_name]['model']

joblib.dump(best_model, models_dir / 'best_churn_model.joblib')
joblib.dump(preprocessor, models_dir / 'preprocessor.joblib')

print(f"Saved best model: {best_model_name}")
